# 06 · Topographic organization metric

A *type* of metric that scores a model's spatial unit layout against cortical topography (TDANN / TopoLM), via the correlation-vs-distance profile — not predictivity.

> Run on EC2 (GPU + Brain-Score data). Do **not** run on a laptop.


In [ ]:
import numpy as np
from scipy.ndimage import gaussian_filter
from brainscore.metrics.topographic import (spatial_smoothness,
    correlation_distance_profile, topographic_alignment)
def grid_pos(g):
    r,c=np.meshgrid(np.arange(g),np.arange(g),indexing='ij')
    return np.stack([r.ravel(),c.ravel()],-1)/(g-1)
def topo(n=60,g=16,seed=0,sig=2.0):
    rng=np.random.RandomState(seed); pos=grid_pos(g)
    R=np.stack([gaussian_filter(rng.randn(g,g),sig,mode='wrap').ravel() for _ in range(n)])
    return R,pos
def rand(n=60,g=16,seed=0):
    rng=np.random.RandomState(seed); return rng.randn(n,g*g),grid_pos(g)

## Topographic model has high spatial smoothness; random ~0

In [ ]:
tr,tp=topo(); rr,rp=rand()
print('topographic smoothness:', round(spatial_smoothness(tr,tp),3))
print('random smoothness:     ', round(spatial_smoothness(rr,rp),3))

## Correlation-vs-distance profile (the descriptor)

In [ ]:
import matplotlib.pyplot as plt
cx,cy,_=correlation_distance_profile(tr,tp,n_bins=12)
rx,ry,_=correlation_distance_profile(rr,rp,n_bins=12)
plt.plot(cx,cy,'-o',label='topographic'); plt.plot(rx,ry,'-o',label='random')
plt.xlabel('normalized distance'); plt.ylabel('mean response corr'); plt.legend()
plt.savefig('topo_profile.png',dpi=120,bbox_inches='tight')

On fMRI, the brain's profile is computed from voxel surface coordinates and `TopographicMetric` correlates the two profiles — high when the model is organized like cortex.